# Data Cleaning and Preprocessing

This notebook performs data cleaning on the raw ATP match, ranking, and player datasets. The objective is to construct a consistent, reliable dataset suitable for feature engineering and predictive modelling.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load Combined Raw Data

The combined raw match, ranking, and player datasets produced in the data loading stage are loaded here for cleaning and preprocessing.

In [2]:
DATA_DIR = Path("../data")
COMBINED_DIR = DATA_DIR / "combined"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

matches_path = COMBINED_DIR / "matches_combined_raw.csv"
rankings_path = COMBINED_DIR / "rankings_combined_raw.csv"
players_path = COMBINED_DIR / "players_raw.csv"

df_matches = pd.read_csv(matches_path, parse_dates=["tourney_date"])
df_rankings = pd.read_csv(rankings_path, parse_dates=["ranking_date"])
df_players = pd.read_csv(players_path, parse_dates=["dob"])

print("Matches:", df_matches.shape)
print("Rankings:", df_rankings.shape)
print("Players:", df_players.shape)

Matches: (33517, 49)
Rankings: (1340901, 4)
Players: (65989, 8)


/var/folders/wv/tbs_c8x12sg2k57wmnt893b80000gn/T/ipykernel_29831/1494238979.py:12: DtypeWarning: Columns (0: wikidata_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_players = pd.read_csv(players_path, parse_dates=["dob"])


## 2. Define Dataset Scope

The study focuses on ATP singles matches from the modern professional tour. The cleaning process therefore applies the following scope restrictions:

- Match dates from 2013 onwards
- Main ATP tour events only
- Completed singles matches only
- Standard modern surfaces only

In [3]:
print("Tournament levels:")
print(df_matches["tourney_level"].value_counts(dropna=False))

print("\nSurfaces:")
print(df_matches["surface"].value_counts(dropna=False))

print("\nDate range:")
print(df_matches["tourney_date"].min(), "to", df_matches["tourney_date"].max())

Tournament levels:
tourney_level
A    17922
M     6555
G     5969
D     2765
F      242
O       64
Name: count, dtype: int64

Surfaces:
surface
Hard      19707
Clay      10196
Grass      3535
NaN          53
Carpet       26
Name: count, dtype: int64

Date range:
2012-12-30 00:00:00 to 2024-12-18 00:00:00


## 3. Filter to Relevant ATP Events

Only the following tournament levels are retained:

- `G` = Grand Slams
- `M` = Masters 1000
- `A` = ATP Tour events
- `F` = Tour Finals

Other event types such as Davis Cup and miscellaneous events are excluded because they differ structurally from standard ATP tournament matches and may distort model comparability.

In [4]:
keep_levels = ["G", "M", "A", "F"]

matches_before = df_matches.shape[0]

df_matches = df_matches[
    (df_matches["tourney_date"] >= "2013-01-01") &
    (df_matches["tourney_level"].isin(keep_levels))
].copy()

print("Rows before filtering:", matches_before)
print("Rows after filtering:", df_matches.shape[0])
print("\nRemaining tournament levels:")
print(df_matches["tourney_level"].value_counts())

Rows before filtering: 33517
Rows after filtering: 30603

Remaining tournament levels:
tourney_level
A    17837
M     6555
G     5969
F      242
Name: count, dtype: int64


## 4. Clean Surface Information

Matches with missing surface values are removed. Carpet court matches are also excluded due to their very small sample size and limited relevance in the modern ATP era. Retaining only hard, clay, and grass surfaces improves consistency for surface-based analysis and surface-specific Elo modelling.

In [5]:
matches_before = df_matches.shape[0]

df_matches = df_matches[df_matches["surface"].isin(["Hard", "Clay", "Grass"])].copy()

print("Rows before surface filtering:", matches_before)
print("Rows after surface filtering:", df_matches.shape[0])
print("\nRemaining surfaces:")
print(df_matches["surface"].value_counts())

Rows before surface filtering: 30603
Rows after surface filtering: 30603

Remaining surfaces:
surface
Hard     17792
Clay      9336
Grass     3475
Name: count, dtype: int64


## 5. Remove Incomplete Matches

Matches decided by retirement, walkover, or other incomplete circumstances are removed. These matches often contain missing or distorted serve statistics and are therefore unsuitable for feature engineering based on match-level performance indicators.

In [6]:
df_matches["score"] = df_matches["score"].astype(str)

bad_score_mask = df_matches["score"].str.contains(
    "RET|W/O|Walkover|DEF|Def\\.|Abandoned|Cancelled",
    case=False,
    na=False,
    regex=True
)

print("Potential incomplete matches:", bad_score_mask.sum())
df_matches.loc[bad_score_mask, ["tourney_name", "tourney_date", "winner_name", "loser_name", "score"]].head(10)

Potential incomplete matches: 1078


,tourney_name,tourney_date,winner_name,loser_name,score
96,Auckland,2013-01-07,Alejandro Falla,Grega Zemlja,6-4 3-1 RET
112,Sydney,2013-01-07,Ryan Harrison,Roberto Bautista Agut,2-1 RET
125,Sydney,2013-01-07,Julien Benneteau,Radek Stepanek,RET
174,Australian Open,2013-01-14,Marcel Granollers,Grega Zemlja,7-6(6) 7-6(5) 1-0 RET
205,Australian Open,2013-01-14,Sam Querrey,Brian Baker,6-7(2) 1-1 RET
206,Australian Open,2013-01-14,Stan Wawrinka,Tobias Kamke,6-3 7-6(4) RET
254,Australian Open,2013-01-14,Nicolas Almagro,Janko Tipsarevic,6-2 5-1 RET
394,Montpellier,2013-02-04,Julien Benneteau,Adrian Ungur,6-3 4-2 RET
407,Montpellier,2013-02-04,Jan Hajek,Viktor Troicki,6-0 4-2 RET
439,Santiago,2013-02-04,Carlos Berlocq,Guillaume Rufin,W/O


In [7]:
matches_before = df_matches.shape[0]

df_matches = df_matches[~bad_score_mask].copy()

print("Rows before removing incomplete matches:", matches_before)
print("Rows after removing incomplete matches:", df_matches.shape[0])

Rows before removing incomplete matches: 30603
Rows after removing incomplete matches: 29525


## 6. Remove Matches Missing Critical Fields

Feature engineering in this project depends on complete match-level serve statistics and valid player identifiers. Matches missing these critical fields are removed.

Critical fields include:
- winner and loser player IDs
- surface
- core serve statistics for both players

In [8]:
critical_match_cols = [
    "winner_id", "loser_id", "surface",
    "w_svpt", "w_1stIn", "w_1stWon", "w_2ndWon", "w_SvGms", "w_bpSaved", "w_bpFaced",
    "l_svpt", "l_1stIn", "l_1stWon", "l_2ndWon", "l_SvGms", "l_bpSaved", "l_bpFaced"
]

missing_critical = df_matches[critical_match_cols].isnull().any(axis=1)

print("Rows with missing critical values:", missing_critical.sum())

matches_before = df_matches.shape[0]
df_matches = df_matches[~missing_critical].copy()

print("Rows before dropping missing critical fields:", matches_before)
print("Rows after dropping missing critical fields:", df_matches.shape[0])

Rows with missing critical values: 117
Rows before dropping missing critical fields: 29525
Rows after dropping missing critical fields: 29408


## 7. Validate Statistical Consistency

Basic logical checks are performed to ensure that the retained match statistics are internally consistent. For example, first-serve points won cannot exceed first serves in, and break points saved cannot exceed break points faced.

In [9]:
consistency_mask = (
    (df_matches["w_1stWon"] <= df_matches["w_1stIn"]) &
    (df_matches["l_1stWon"] <= df_matches["l_1stIn"]) &
    (df_matches["w_bpSaved"] <= df_matches["w_bpFaced"]) &
    (df_matches["l_bpSaved"] <= df_matches["l_bpFaced"]) &
    (df_matches["winner_age"] > 0) &
    (df_matches["loser_age"] > 0)
)

print("Inconsistent rows:", (~consistency_mask).sum())

matches_before = df_matches.shape[0]
df_matches = df_matches[consistency_mask].copy()

print("Rows before consistency filtering:", matches_before)
print("Rows after consistency filtering:", df_matches.shape[0])

Inconsistent rows: 0
Rows before consistency filtering: 29408
Rows after consistency filtering: 29408


## 8. Sort Chronologically and Remove Duplicates

Chronological ordering is essential for later rolling feature construction and time-aware model evaluation. Duplicate matches are also removed where necessary.

In [10]:
print("Duplicate rows:", df_matches.duplicated().sum())

df_matches = df_matches.drop_duplicates().copy()

df_matches = df_matches.sort_values(
    by=["tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

print("Final cleaned match shape:", df_matches.shape)
df_matches.head()

Duplicate rows: 0
Final cleaned match shape: (29408, 49)


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2013-301,Auckland,Hard,28,A,2013-01-07,1,104229,NaN,NaN,...,40.0,29.0,14.0,13.0,2.0,7.0,60.0,754.0,43.0,937.0
1,2013-301,Auckland,Hard,28,A,2013-01-07,2,105041,NaN,NaN,...,45.0,27.0,12.0,9.0,6.0,9.0,48.0,872.0,57.0,769.0
2,2013-301,Auckland,Hard,28,A,2013-01-07,3,105064,8.0,NaN,...,44.0,27.0,5.0,10.0,0.0,5.0,32.0,1112.0,50.0,843.0
3,2013-301,Auckland,Hard,28,A,2013-01-07,4,104997,NaN,Q,...,43.0,38.0,20.0,15.0,3.0,5.0,70.0,706.0,54.0,800.0
4,2013-301,Auckland,Hard,28,A,2013-01-07,5,104792,NaN,WC,...,53.0,44.0,17.0,15.0,4.0,7.0,99.0,555.0,65.0,730.0


## 9. Clean Ranking Data

Ranking history is required for engineered features such as career-high ranking, ranking change, and ranking volatility. Duplicate records are removed and the dataset is sorted chronologically.

In [11]:
print("Ranking duplicates:", df_rankings.duplicated(subset=["ranking_date", "player"]).sum())

df_rankings = df_rankings.drop_duplicates(subset=["ranking_date", "player"]).copy()
df_rankings = df_rankings.sort_values(by=["ranking_date", "player"]).reset_index(drop=True)

print("Cleaned rankings shape:", df_rankings.shape)
df_rankings.head()

Ranking duplicates: 613
Cleaned rankings shape: (1340288, 4)


,ranking_date,rank,player,points
0,2010-01-04,1596,101549,1.0
1,2010-01-04,303,101868,138.0
2,2010-01-04,1095,102106,5.0
3,2010-01-04,68,102148,645.0
4,2010-01-04,505,102179,58.0


## 10. Clean Player Metadata

The player file is retained as a supporting metadata table. Only duplicate rows are removed at this stage. Missing height or date-of-birth values are tolerated because these fields are not essential for the initial match-level feature set.

In [12]:
print("Player duplicates:", df_players.duplicated(subset=["player_id"]).sum())

df_players = df_players.drop_duplicates(subset=["player_id"]).copy()

print("Cleaned players shape:", df_players.shape)
df_players.head()

Player duplicates: 0
Cleaned players shape: (65989, 8)


,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
0,100001,Gardnar,Mulloy,R,1913-11-22,USA,185.0,Q54544
1,100002,Pancho,Segura,R,1921-06-20,ECU,168.0,Q54581
2,100003,Frank,Sedgman,R,1927-10-02,AUS,180.0,Q962049
3,100004,Giuseppe,Merlo,R,1927-10-11,ITA,NaN,Q1258752
4,100005,Richard,Gonzalez,R,1928-05-09,USA,188.0,Q53554


## 10.1 Inspect Duplicate Ranking Entries

Ranking history should contain one entry per player per ranking date. Duplicate player-date rows are inspected here to confirm whether they are exact duplicates or conflicting records.

In [13]:
dupes = df_rankings[df_rankings.duplicated(subset=["ranking_date", "player"], keep=False)].copy()

print("Duplicate rows:", len(dupes))
print("Unique duplicated player-date pairs:", dupes.drop_duplicates(subset=["ranking_date", "player"]).shape[0])

dupes = dupes.sort_values(["player", "ranking_date", "rank", "points"])
dupes.head(20)

dupe_groups = (
    dupes.groupby(["ranking_date", "player"])
    .agg(
        n_rows=("player", "size"),
        n_unique_rank=("rank", "nunique"),
        n_unique_points=("points", "nunique"),
    )
    .reset_index()
)

print(dupe_groups.head(20))

print("\nGroups with conflicting rank values:")
print((dupe_groups["n_unique_rank"] > 1).sum())

print("\nGroups with conflicting points values:")
print((dupe_groups["n_unique_points"] > 1).sum())

conflicting = dupe_groups[
    (dupe_groups["n_unique_rank"] > 1) | (dupe_groups["n_unique_points"] > 1)
].copy()

print("Conflicting duplicate groups:", conflicting.shape[0])

if not conflicting.empty:
    sample_conflicts = dupes.merge(
        conflicting[["ranking_date", "player"]],
        on=["ranking_date", "player"],
        how="inner"
    ).sort_values(["player", "ranking_date", "rank", "points"])
    display(sample_conflicts.head(20))

Duplicate rows: 0
Unique duplicated player-date pairs: 0
Empty DataFrame
Columns: [ranking_date, player, n_rows, n_unique_rank, n_unique_points]
Index: []

Groups with conflicting rank values:
0

Groups with conflicting points values:
0
Conflicting duplicate groups: 0


## 11. Final Dataset Summary

The cleaned datasets are summarised below before being saved for feature engineering.

In [14]:
print("Cleaned match dataset:")
print(df_matches.shape)
print(df_matches["tourney_level"].value_counts())
print(df_matches["surface"].value_counts())
print(df_matches["best_of"].value_counts())

print("\nCleaned ranking dataset:")
print(df_rankings.shape)

print("\nCleaned player dataset:")
print(df_players.shape)

Cleaned match dataset:
(29408, 49)
tourney_level
A    17112
M     6337
G     5723
F      236
Name: count, dtype: int64
surface
Hard     17011
Clay      9035
Grass     3362
Name: count, dtype: int64
best_of
3    23641
5     5767
Name: count, dtype: int64

Cleaned ranking dataset:
(1340288, 4)

Cleaned player dataset:
(65989, 8)


## 12. Save Cleaned Data

The cleaned match, ranking, and player datasets are saved for use in the next stage of the pipeline: feature engineering.

In [15]:
df_matches.to_csv(PROCESSED_DIR / "matches_clean.csv", index=False)
df_rankings.to_csv(PROCESSED_DIR / "rankings_clean.csv", index=False)
df_players.to_csv(PROCESSED_DIR / "players_clean.csv", index=False)

print("Saved cleaned files:")
print(" -", (PROCESSED_DIR / "matches_clean.csv").resolve())
print(" -", (PROCESSED_DIR / "rankings_clean.csv").resolve())
print(" -", (PROCESSED_DIR / "players_clean.csv").resolve())

Saved cleaned files:
 - /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/data/processed/matches_clean.csv
 - /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/data/processed/rankings_clean.csv
 - /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/data/processed/players_clean.csv
